# RAID v3: Top-k Accuracy Memory Curves

**Problem with perplexity:** LMs reduce perplexity with more context even on random text, by calibrating their output distribution. This instrument bias contaminates the memory curve shape.

**Solution:** Replace perplexity with **top-k accuracy** — the fraction of target tokens where the correct next token is in the model's top-k predictions. This measures whether the model can actually *predict* what comes next, not just calibrate its uncertainty.

**Expected results:**
- **Random text:** ~0% accuracy at all context lengths (flat line) — no amount of context helps predict random tokens
- **Shuffled text:** Low accuracy, slight improvement with context (token frequency helps a little)
- **Real text:** Accuracy increases with context, front-loaded by local coherence structure
- The normalized curve for real text should now cleanly reflect text structure with no instrument bias

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.integrate import trapezoid
from pathlib import Path
import json, math, time, gc, os, torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("Imports OK")

In [ ]:
IN_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DATA = Path("/content/drive/MyDrive/LRTIA/Data/raid_sampled")
    DRIVE_RESULTS = Path("/content/drive/MyDrive/LRTIA/Results/RAID_v3")
    if (DRIVE_DATA / "raid_corpus.jsonl").exists():
        DATA_DIR = DRIVE_DATA
        print(f"Found corpus on Drive: {DATA_DIR}")
    else:
        LOCAL_DATA = Path("/content/data/raid_sampled")
        if not (LOCAL_DATA / "raid_corpus.jsonl").exists():
            print("Corpus not found on Drive. Upload raid_corpus.jsonl:")
            LOCAL_DATA.mkdir(parents=True, exist_ok=True)
            from google.colab import files
            uploaded = files.upload()
            for fname in uploaded:
                with open(LOCAL_DATA / fname, 'wb') as f:
                    f.write(uploaded[fname])
        DATA_DIR = LOCAL_DATA
        print(f"Using local corpus: {DATA_DIR}")
    BASE_DIR = DRIVE_RESULTS
    BASE_DIR.mkdir(parents=True, exist_ok=True)
else:
    BASE_DIR = Path("../results/raid_v3")
    DATA_DIR = Path("../data/raid_sampled")
    BASE_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_DIR: {DATA_DIR}")
print(f"BASE_DIR: {BASE_DIR}")

MODEL_NAME = "mistralai/Mistral-7B-v0.1"
USE_4BIT = True

WINDOWS = [4, 8, 12, 16, 24, 32, 48, 64, 96, 128]
BURN_IN = 128
MAX_SCORE_TOKENS = 64
BUFFER = 10
MIN_TOKENS = BURN_IN + MAX_SCORE_TOKENS + BUFFER

TOP_K_VALUES = [1, 5, 10, 50, 100]  # compute accuracy at multiple k values
N_RANDOM_CONTROLS = 30
RANDOM_SEED = 42

DOMAINS = ['abstracts', 'books', 'news', 'poetry', 'recipes', 'reddit', 'reviews', 'wiki']
COLORS_POP = {'human': '#3498db', 'ai': '#e74c3c'}
COLORS_CTRL = {'human': '#3498db', 'ai': '#e74c3c', 'shuffled': '#2ecc71', 'uniform': '#9b59b6'}

print(f"Windows: {WINDOWS}")
print(f"Top-k values: {TOP_K_VALUES}")
print(f"Min tokens: {MIN_TOKENS}")

In [ ]:
corpus_path = DATA_DIR / "raid_corpus.jsonl"
corpus = []
with open(corpus_path) as f:
    for line in f:
        corpus.append(json.loads(line))

print(f"Loaded {len(corpus)} documents")
print(f"\nBy domain x population:")
for d in DOMAINS:
    nh = sum(1 for c in corpus if c['domain'] == d and c['population'] == 'human')
    na = sum(1 for c in corpus if c['domain'] == d and c['population'] == 'ai')
    print(f"  {d:<15} human={nh}, ai={na}")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto")
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
model.eval()
print("Model loaded")

In [ ]:
@torch.no_grad()
def compute_metrics_on_region(token_ids, target_start, target_end, top_k_values=[1, 5, 10, 50, 100]):
    """Compute perplexity AND top-k accuracy on tokens in [target_start, target_end)."""
    if target_end > len(token_ids):
        target_end = len(token_ids)
    if target_start >= target_end - 1:
        return None

    input_ids = torch.tensor([token_ids], device=model.device)
    outputs = model(input_ids)
    logits = outputs.logits[0]

    total_loss = 0.0
    topk_hits = {k: 0 for k in top_k_values}
    total_rank = 0.0
    count = 0

    max_k = max(top_k_values)

    for i in range(target_start, target_end - 1):
        target_token = token_ids[i + 1]
        token_logits = logits[i]

        # Perplexity
        log_probs = torch.log_softmax(token_logits, dim=-1)
        total_loss += -log_probs[target_token].item()

        # Top-k accuracy (single topk call covers all k values)
        top_ids = torch.topk(token_logits, max_k).indices
        for k in top_k_values:
            if target_token in top_ids[:k]:
                topk_hits[k] += 1

        # Rank: count how many tokens have higher logit (avoids full argsort)
        rank = (token_logits > token_logits[target_token]).sum().item()
        total_rank += rank

        count += 1

    # Free GPU memory
    del outputs, logits
    torch.cuda.empty_cache()

    if count == 0:
        return None

    return {
        'ppl': math.exp(total_loss / count),
        'mean_rank': total_rank / count,
        **{f'top{k}_acc': topk_hits[k] / count for k in top_k_values},
        'count': count,
    }


def compute_memory_curve(token_ids, windows, burn_in, max_score_tokens):
    """Compute all metrics at each context window size."""
    n_tokens = len(token_ids)
    if n_tokens <= burn_in:
        return None
    target_end = min(n_tokens, burn_in + max_score_tokens)

    results = {}
    for W in windows:
        context_start = max(0, burn_in - W)
        actual_context = burn_in - context_start
        if actual_context < 4:
            continue
        truncated = token_ids[context_start:target_end]
        metrics = compute_metrics_on_region(truncated, actual_context, len(truncated), TOP_K_VALUES)
        if metrics is not None and not math.isinf(metrics['ppl']):
            results[W] = metrics

    return results if len(results) >= 3 else None


def compute_half_life(values_by_W, metric_key, increasing=True):
    """Half-life for any metric. increasing=True for accuracy (goes up), False for perplexity (goes down)."""
    if len(values_by_W) < 2:
        return float('nan')
    items = sorted(values_by_W.items())
    windows = np.array([x[0] for x in items])
    vals = np.array([x[1][metric_key] for x in items])

    if increasing:
        total_benefit = vals[-1] - vals[0]
        if total_benefit <= 0:
            return float('nan')
        target = vals[0] + 0.5 * total_benefit
        for i in range(len(vals) - 1):
            if vals[i] <= target <= vals[i + 1]:
                frac = (target - vals[i]) / (vals[i + 1] - vals[i])
                return windows[i] + frac * (windows[i + 1] - windows[i])
    else:
        total_benefit = vals[0] - vals[-1]
        if total_benefit <= 0:
            return float('nan')
        target = vals[0] - 0.5 * total_benefit
        for i in range(len(vals) - 1):
            if vals[i] >= target >= vals[i + 1]:
                frac = (vals[i] - target) / (vals[i] - vals[i + 1])
                return windows[i] + frac * (windows[i + 1] - windows[i])

    return windows[-1]


def extract_row(doc, curve_results, n_tokens):
    """Extract all summary metrics from a curve result dict."""
    row = {
        'doc_id': doc['doc_id'],
        'domain': doc['domain'],
        'model': doc['model'],
        'population': doc['population'],
        'token_count': n_tokens,
    }

    for W, metrics in sorted(curve_results.items()):
        row[f'ppl_W{W}'] = metrics['ppl']
        row[f'rank_W{W}'] = metrics['mean_rank']
        for k in TOP_K_VALUES:
            row[f'top{k}_W{W}'] = metrics[f'top{k}_acc']

    row['hl_ppl'] = compute_half_life(curve_results, 'ppl', increasing=False)
    row['hl_rank'] = compute_half_life(curve_results, 'mean_rank', increasing=False)
    for k in TOP_K_VALUES:
        row[f'hl_top{k}'] = compute_half_life(curve_results, f'top{k}_acc', increasing=True)

    sorted_W = sorted(curve_results.keys())
    top10_vals = np.array([curve_results[w]['top10_acc'] for w in sorted_W])
    row['top10_min'] = top10_vals[0]
    row['top10_max'] = top10_vals[-1]
    row['top10_delta'] = top10_vals[-1] - top10_vals[0]
    row['top10_auc'] = trapezoid(top10_vals, sorted_W)

    early_W = [w for w in sorted_W if w <= 32]
    if len(early_W) >= 2:
        early_vals = np.array([curve_results[w]['top10_acc'] for w in early_W])
        delta_early = early_vals - early_vals[0]
        row['top10_auc_early'] = trapezoid(delta_early, early_W)
    full_delta = top10_vals - top10_vals[0]
    row['top10_auc_benefit'] = trapezoid(full_delta, sorted_W)
    if row['top10_auc_benefit'] > 0 and 'top10_auc_early' in row:
        row['top10_early_fraction'] = row['top10_auc_early'] / row['top10_auc_benefit']

    return row


print("Functions defined")

In [ ]:
results_path = BASE_DIR / "essay_results_v3.csv"

if results_path.exists():
    df = pd.read_csv(results_path)
    print(f"Loaded existing results: {len(df)} rows")
else:
    results = []
    skipped = 0
    for doc in tqdm(corpus, desc="Essays"):
        token_ids = tokenizer.encode(doc["text"], add_special_tokens=False)
        n_tokens = len(token_ids)
        if n_tokens < MIN_TOKENS:
            skipped += 1
            continue
        curve = compute_memory_curve(token_ids, WINDOWS, BURN_IN, MAX_SCORE_TOKENS)
        if curve is None:
            skipped += 1
            continue
        row = extract_row(doc, curve, n_tokens)
        results.append(row)

    df = pd.DataFrame(results)
    df.to_csv(results_path, index=False)
    print(f"Processed {len(df)} essays ({skipped} skipped)")
    print(f"Saved to {results_path}")

print(f"Human: {len(df[df.population == 'human'])}, AI: {len(df[df.population == 'ai'])}")

In [ ]:
controls_path = BASE_DIR / "control_results_v3.csv"

if controls_path.exists():
    df_ctrl = pd.read_csv(controls_path)
    print(f"Loaded existing controls: {len(df_ctrl)} rows")
else:
    rng = np.random.RandomState(RANDOM_SEED)
    human_docs = [d for d in corpus if d['population'] == 'human']
    sample_docs = rng.choice(human_docs, size=min(N_RANDOM_CONTROLS, len(human_docs)), replace=False)

    ctrl_rows = []

    # Shuffled tokens
    rng_s = np.random.RandomState(RANDOM_SEED)
    for i, doc in enumerate(tqdm(sample_docs, desc="Shuffled")):
        token_ids = tokenizer.encode(doc["text"], add_special_tokens=False)
        if len(token_ids) < MIN_TOKENS:
            continue
        shuffled = list(token_ids)
        rng_s.shuffle(shuffled)
        curve = compute_memory_curve(shuffled, WINDOWS, BURN_IN, MAX_SCORE_TOKENS)
        if curve is None:
            continue
        ctrl_doc = {'doc_id': f'shuffled_{i:03d}', 'domain': 'shuffled', 'model': 'shuffled', 'population': 'shuffled'}
        ctrl_rows.append(extract_row(ctrl_doc, curve, len(token_ids)))

    # Uniform random tokens
    rng_u = np.random.RandomState(RANDOM_SEED + 1)
    vocab_size = tokenizer.vocab_size
    for i, doc in enumerate(tqdm(sample_docs, desc="Uniform")):
        token_ids = tokenizer.encode(doc["text"], add_special_tokens=False)
        if len(token_ids) < MIN_TOKENS:
            continue
        uniform_ids = rng_u.randint(0, vocab_size, size=len(token_ids)).tolist()
        curve = compute_memory_curve(uniform_ids, WINDOWS, BURN_IN, MAX_SCORE_TOKENS)
        if curve is None:
            continue
        ctrl_doc = {'doc_id': f'uniform_{i:03d}', 'domain': 'uniform', 'model': 'uniform', 'population': 'uniform'}
        ctrl_rows.append(extract_row(ctrl_doc, curve, len(token_ids)))

    df_ctrl = pd.DataFrame(ctrl_rows)
    df_ctrl.to_csv(controls_path, index=False)
    print(f"Controls: {len(df_ctrl)} rows")
    print(f"Saved to {controls_path}")

print(f"Shuffled: {len(df_ctrl[df_ctrl.population == 'shuffled'])}, Uniform: {len(df_ctrl[df_ctrl.population == 'uniform'])}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for row_idx, (k, metric_label) in enumerate([(10, 'Top-10 Accuracy'), (1, 'Top-1 Accuracy')]):
    cols = [f'top{k}_W{w}' for w in WINDOWS]

    # Raw accuracy curves
    ax = axes[row_idx, 0]
    for pop, label, color, ls, lw in [
        ('human', 'Human', COLORS_CTRL['human'], '-', 2.5),
        ('ai', 'AI', COLORS_CTRL['ai'], '--', 1.5),
    ]:
        sub = df[df.population == pop]
        means = np.array([sub[c].mean() for c in cols])
        ax.plot(WINDOWS, means, f'o{ls}', color=color, linewidth=lw, markersize=4, label=label)

    for pop, label, color in [('shuffled', 'Shuffled', COLORS_CTRL['shuffled']), ('uniform', 'Uniform', COLORS_CTRL['uniform'])]:
        sub = df_ctrl[df_ctrl.population == pop]
        if len(sub) > 0:
            avail = [c for c in cols if c in sub.columns]
            means = np.array([sub[c].mean() for c in avail])
            ax.plot(WINDOWS[:len(means)], means, 's:', color=color, linewidth=2, markersize=5, label=label)

    ax.set_xscale('log', base=2)
    ax.set_xlabel('Context Window')
    ax.set_ylabel(metric_label)
    ax.set_title(f'Raw {metric_label}', fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

    # Normalized curves
    ax = axes[row_idx, 1]
    for pop, label, color, ls, lw, src_df in [
        ('human', 'Human', COLORS_CTRL['human'], '-', 2.5, df),
        ('ai', 'AI', COLORS_CTRL['ai'], '--', 1.5, df),
        ('shuffled', 'Shuffled', COLORS_CTRL['shuffled'], ':', 2, df_ctrl),
        ('uniform', 'Uniform', COLORS_CTRL['uniform'], ':', 2, df_ctrl),
    ]:
        sub = src_df[src_df.population == pop]
        if len(sub) == 0:
            continue
        avail = [c for c in cols if c in sub.columns]
        means = np.array([sub[c].mean() for c in avail])
        total = means[-1] - means[0]
        if total > 0:
            norm = (means - means[0]) / total
            marker = 'o' if pop in ['human', 'ai'] else 's'
            ax.plot(WINDOWS[:len(norm)], norm, f'{marker}{ls}', color=color, linewidth=lw, markersize=4, label=label)

    ax.axhline(0.5, color='gray', linestyle=':', alpha=0.4)
    ax.plot([WINDOWS[0], WINDOWS[-1]], [0, 1], 'k--', alpha=0.2, label='Linear')
    ax.set_xscale('log', base=2)
    ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel('Context Window')
    ax.set_ylabel('Fraction of Total Benefit')
    ax.set_title(f'Normalized {metric_label}', fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

    # Perplexity comparison (same data, old metric)
    ax = axes[row_idx, 2]
    ppl_cols = [f'ppl_W{w}' for w in WINDOWS]
    for pop, label, color, ls, lw, src_df in [
        ('human', 'Human', COLORS_CTRL['human'], '-', 2.5, df),
        ('ai', 'AI', COLORS_CTRL['ai'], '--', 1.5, df),
        ('shuffled', 'Shuffled', COLORS_CTRL['shuffled'], ':', 2, df_ctrl),
        ('uniform', 'Uniform', COLORS_CTRL['uniform'], ':', 2, df_ctrl),
    ]:
        sub = src_df[src_df.population == pop]
        if len(sub) == 0:
            continue
        avail = [c for c in ppl_cols if c in sub.columns]
        means = np.array([sub[c].mean() for c in avail])
        total = means[0] - means[-1]
        if total > 0:
            norm = (means[0] - means) / total
            marker = 'o' if pop in ['human', 'ai'] else 's'
            ax.plot(WINDOWS[:len(norm)], norm, f'{marker}{ls}', color=color, linewidth=lw, markersize=4, label=label)

    ax.axhline(0.5, color='gray', linestyle=':', alpha=0.4)
    ax.plot([WINDOWS[0], WINDOWS[-1]], [0, 1], 'k--', alpha=0.2, label='Linear')
    ax.set_xscale('log', base=2)
    ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel('Context Window')
    ax.set_ylabel('Fraction of Total Benefit')
    ax.set_title(f'Normalized Perplexity (comparison)', fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

plt.suptitle('Top-k Accuracy vs Perplexity: Do Controls Look Different?', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig1_topk_vs_ppl.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top-10 accuracy curves by genre (human vs AI)
top10_cols = [f'top10_W{w}' for w in WINDOWS]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for idx, domain in enumerate(DOMAINS):
    ax = axes[idx // 4, idx % 4]
    for pop, ls, lw in [('human', '-', 2.5), ('ai', '--', 1.5)]:
        sub = df[(df.domain == domain) & (df.population == pop)]
        if len(sub) == 0:
            continue
        means = np.array([sub[c].mean() for c in top10_cols])
        total = means[-1] - means[0]
        if total > 0:
            norm = (means - means[0]) / total
            ax.plot(WINDOWS, norm, marker='o', linestyle=ls, linewidth=lw,
                    color=COLORS_POP[pop], markersize=4, label=pop)

    ax.axhline(0.5, color='gray', linestyle=':', alpha=0.4)
    ax.set_title(domain, fontsize=12, fontweight='bold')
    ax.set_xscale('log', base=2)
    ax.set_ylim(-0.05, 1.05)
    if idx % 4 == 0:
        ax.set_ylabel('Fraction of Total Accuracy Gain')
    if idx >= 4:
        ax.set_xlabel('Context Window')
    ax.grid(True, alpha=0.2)
    ax.legend(fontsize=8)

plt.suptitle('Normalized Top-10 Accuracy Curves by Genre: Human vs AI',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig2_topk_by_genre.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Summary statistics
print("="*70)
print("SUMMARY: Top-k Accuracy Memory Curves")
print("="*70)

print("\n--- Raw accuracy at W4 and W128 ---")
for k in TOP_K_VALUES:
    cols_k = [f'top{k}_W{w}' for w in WINDOWS]
    print(f"\nTop-{k}:")
    print(f"  {'Condition':<12} {'W4':>8} {'W128':>8} {'Delta':>8} {'Half-life':>10}")
    print(f"  {'-'*42}")
    for pop, label, src_df in [('human', 'Human', df), ('ai', 'AI', df),
                                ('shuffled', 'Shuffled', df_ctrl), ('uniform', 'Uniform', df_ctrl)]:
        sub = src_df[src_df.population == pop]
        if len(sub) == 0:
            continue
        w4 = sub[f'top{k}_W4'].mean() if f'top{k}_W4' in sub.columns else np.nan
        w128 = sub[f'top{k}_W128'].mean() if f'top{k}_W128' in sub.columns else np.nan
        hl = sub[f'hl_top{k}'].mean() if f'hl_top{k}' in sub.columns else np.nan
        print(f"  {label:<12} {w4:>8.3f} {w128:>8.3f} {w128-w4:>8.3f} {hl:>10.1f}")

print("\n\n--- Half-life comparison: Perplexity vs Top-10 ---")
print(f"  {'Condition':<12} {'HL (ppl)':>10} {'HL (top10)':>12}")
print(f"  {'-'*36}")
for pop, label, src_df in [('human', 'Human', df), ('ai', 'AI', df),
                            ('shuffled', 'Shuffled', df_ctrl), ('uniform', 'Uniform', df_ctrl)]:
    sub = src_df[src_df.population == pop]
    if len(sub) == 0:
        continue
    hl_ppl = sub['hl_ppl'].mean() if 'hl_ppl' in sub.columns else np.nan
    hl_top10 = sub['hl_top10'].mean() if 'hl_top10' in sub.columns else np.nan
    print(f"  {label:<12} {hl_ppl:>10.1f} {hl_top10:>12.1f}")

print("\n\n--- Genre universality (Top-10 half-life, human only) ---")
h = df[df.population == 'human']
print(f"  {'Genre':<15} {'HL (top10)':>12} {'Top10 W4':>10} {'Top10 W128':>10}")
print(f"  {'-'*50}")
for d in DOMAINS:
    sub = h[h.domain == d]
    hl = sub['hl_top10'].dropna()
    w4 = sub['top10_W4'].mean()
    w128 = sub['top10_W128'].mean()
    print(f"  {d:<15} {hl.mean():>12.1f} {w4:>10.3f} {w128:>10.3f}")

# ANOVA on top-10 half-life
groups = [h[h.domain == d]['hl_top10'].dropna() for d in DOMAINS if len(h[h.domain == d]['hl_top10'].dropna()) > 2]
if len(groups) >= 2:
    f_val, p_val = stats.f_oneway(*groups)
    print(f"\n  ANOVA (genre effect): F={f_val:.3f}, p={p_val:.4f}")

# Human vs AI
print("\n\n--- Human vs AI (Top-10 half-life) ---")
h_hl = df[df.population == 'human']['hl_top10'].dropna()
a_hl = df[df.population == 'ai']['hl_top10'].dropna()
t, p = stats.ttest_ind(h_hl, a_hl)
d_val = (h_hl.mean() - a_hl.mean()) / np.sqrt((h_hl.std()**2 + a_hl.std()**2) / 2)
print(f"  Human: {h_hl.mean():.1f}, AI: {a_hl.mean():.1f}, d={d_val:.3f}, p={p:.4f}")